# LangChain.js RAG with Oracle AI Database

This notebook shows how to build a JavaScript RAG workflow with LangChain.js and Oracle AI Database.

It focuses on the TypeScript and JavaScript integration package `@oracle/langchain-oracledb`. The notebook uses a Python Jupyter kernel only to orchestrate the tutorial. The actual JavaScript implementation lives in `langchain_rag.mjs`, runs with Node.js, and can be moved into a TypeScript backend with the same package imports and OracleVS pattern.

**What this notebook demonstrates**

- Connect from Node.js to FreeSQL, Autonomous AI Database, or local Oracle Database.
- Validate the schema before creating vector-store objects.
- Store documents, embeddings, and JSON metadata with `OracleVS`.
- Inspect the Oracle table created by the vector store.
- Run similarity search, metadata-filtered search, and MMR retrieval.
- Show where vector index creation fits when the database environment supports it.
- Generate a grounded RAG answer with a chat model from retrieved OracleVS context.

## Architecture at a Glance

```text
                 +-------------------------------+
                 | JavaScript / TypeScript app   |
                 | Node.js + LangChain.js        |
                 +---------------+---------------+
                                 |
                                 | uses
                                 v
                 +-------------------------------+
                 | @oracle/langchain-oracledb    |
                 | OracleVS vector store         |
                 +---------------+---------------+
                                 |
                                 | stores and queries
                                 v
+----------------------------------------------------------------+
|                     Oracle AI Database                         |
|  VECTOR embeddings + document text + JSON metadata + indexes   |
+-------------------------------+--------------------------------+
                                |
                                | returns retrieved context
                                v
                 +-------------------------------+
                 | Grounded RAG response         |
                 +-------------------------------+
```

`OracleVS` is the main integration point. The application keeps the LangChain developer experience, while Oracle AI Database stores the text, embedding vectors, and metadata used for retrieval.

## What the Oracle LangChain.js Package Provides

The Oracle JavaScript integration package connects LangChain.js applications to Oracle AI Database. Its surface includes vector storage, similarity search, document loading, text splitting, in-database embeddings, summarization, and vector index creation.

This notebook focuses the runnable path on the pieces that make the first tutorial strong and easy to verify:

| Capability | Why it matters in a RAG app | Covered here |
| --- | --- | --- |
| `OracleVS` | Stores text, metadata, and embeddings in Oracle AI Database | Runnable end to end |
| Similarity search | Retrieves nearest chunks for a user question | Runnable end to end |
| Metadata filtering | Restricts retrieval using JSON metadata predicates | Runnable end to end |
| MMR retrieval | Balances relevance and diversity in retrieved context | Runnable end to end |
| `createIndex` | Adds HNSW vector indexing when supported by the DB environment | Opt in |
| LLM answer generation | Turns retrieved context into a grounded response | Optional, if `OPENAI_API_KEY` is set |
| `OracleDocLoader`, `OracleTextSplitter`, `OracleEmbeddings`, `OracleSummary` | Production extension paths for ingestion and model operations | Explained at the end |

The runnable JavaScript code is intentionally kept in `langchain_rag.mjs`. The notebook explains and verifies the workflow without hiding a generated application inside notebook cells.

## Database Connection Guide

Use one of these connection paths. The JavaScript demo reads the same `.env` variable names for all three options.

### FreeSQL

Use FreeSQL when you want a hosted Oracle Database schema for a tutorial without setting up a local database.

1. Open [FreeSQL](https://freesql.com).
2. Sign in with your Oracle account.
3. Select **Connect to the Database** from the top navigation.
4. Choose the **NodeJS** tab for this LangChain.js notebook.
5. Copy the generated username, password, and connection string into a private `.env` file in this folder.

<p align="center">
  <img src="./../images/freesql/freesql-interface.png" alt="FreeSQL worksheet interface" width="700" />
</p>

<p align="center">
  <img src="./../images/freesql/freesql-sign-in.png" alt="Oracle sign-in page for FreeSQL" width="700" />
</p>

<p align="center">
  <img src="./../images/freesql/freesql-nodejs-connection.png" alt="FreeSQL NodeJS connection details" width="700" />
</p>

```env
DB_USER=<freesql-user>
DB_PASSWORD=<freesql-password>
DB_CONNECT_STRING=<freesql-connect-descriptor-or-service-url>
LANGCHAIN_JS_TABLE=LC_JS_RAG_DEMO
LANGCHAIN_JS_CREATE_INDEX=false
```

FreeSQL normally works for the main tutorial path because the demo uses regular schema-level table operations. Vector index creation is disabled by default because small hosted schemas may not have enough quota for index auxiliary objects.

### Autonomous AI Database

Use Autonomous AI Database when you want the same LangChain.js workflow against a managed Oracle AI Database environment.

```env
DB_USER=<database-user>
DB_PASSWORD=<database-password>
DB_CONNECT_STRING=<adb-service-name-or-connect-descriptor>
LANGCHAIN_JS_TABLE=LC_JS_RAG_DEMO
LANGCHAIN_JS_CREATE_INDEX=true
```

If your Autonomous Database connection requires a wallet, configure the Node.js Oracle Database driver environment as you normally would for `node-oracledb`, then put the service name or connect descriptor in `DB_CONNECT_STRING`.

### Local Oracle Database

Use a local Oracle Database when you want a repeatable development environment on your machine or in a container.

```env
DB_USER=<local-user>
DB_PASSWORD=<local-password>
DB_CONNECT_STRING=localhost:1521/FREEPDB1
LANGCHAIN_JS_TABLE=LC_JS_RAG_DEMO
LANGCHAIN_JS_CREATE_INDEX=true
```

The schema should be able to create tables and indexes in its own account. If vector index creation fails because of quota or privileges, set `LANGCHAIN_JS_CREATE_INDEX=false` and rerun the demo. Retrieval still works without the vector index for this small tutorial corpus.

## 1. Prepare the Notebook Workspace

This notebook expects to run from the same folder as the JavaScript project files:

- `langchain_rag.mjs`
- `package.json`
- `.env` after you create it locally

The cell below checks that the source files are present and chooses `npm.cmd` automatically on Windows. It deliberately avoids printing local filesystem paths in notebook output.

In [1]:
from pathlib import Path
import json
import os
import platform
import subprocess
import textwrap

from IPython.display import Markdown, display

PROJECT_DIR = Path.cwd()
NODE_APP = PROJECT_DIR / "langchain_rag.mjs"
PACKAGE_JSON = PROJECT_DIR / "package.json"
RESULTS_JSON = PROJECT_DIR / "rag_results.json"
NPM_BIN = "npm.cmd" if platform.system() == "Windows" else "npm"

required_files = [NODE_APP, PACKAGE_JSON]
missing_files = [path.name for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Run this notebook from the tutorial folder. Missing: {missing_files}")

display(Markdown("**Workspace check:** JavaScript source and package metadata found."))

**Workspace check:** JavaScript source and package metadata found.

## 2. Configure the Connection Profile

Create a private `.env` file in this folder. The JavaScript code loads it through `dotenv`, but the notebook never prints credentials and the `.env` file must not be committed.

```env
DB_USER=<database-user>
DB_PASSWORD=<database-password>
DB_CONNECT_STRING=<database-connect-string>
LANGCHAIN_JS_TABLE=LC_JS_RAG_DEMO
LANGCHAIN_JS_CREATE_INDEX=false

# Optional, only for the final grounded-answer step.
OPENAI_API_KEY=<openai-api-key>
LANGCHAIN_JS_LLM_MODEL=gpt-4o-mini
```

The retrieval and OracleVS sections do not require an LLM key because the demo uses deterministic local embeddings. The final RAG answer step is stronger when `OPENAI_API_KEY` is set, but the workflow still completes without it and prints the retrieved context instead.

In [2]:
env_path = PROJECT_DIR / ".env"
if not env_path.exists():
    display(Markdown(
        "**Configuration needed:** create a private `.env` file in this folder with `DB_USER`, `DB_PASSWORD`, and `DB_CONNECT_STRING`."
    ))
else:
    keys = []
    for line in env_path.read_text(encoding="utf-8").splitlines():
        stripped = line.strip()
        if stripped and not stripped.startswith("#") and "=" in stripped:
            keys.append(stripped.split("=", 1)[0])
    safe_keys = ", ".join(f"`{key}`" for key in keys)
    display(Markdown(f"**Configuration found:** {safe_keys}"))

**Configuration found:** `DB_USER`, `DB_PASSWORD`, `DB_CONNECT_STRING`, `LANGCHAIN_JS_TABLE`, `LANGCHAIN_JS_CREATE_INDEX`, `OPENAI_API_KEY`, `LANGCHAIN_JS_LLM_MODEL`

## 3. Inspect the JavaScript Implementation Surface

The implementation is a single runnable Node.js module. Keeping it in a source file matters for a public Developer Hub contribution because reviewers can inspect the JavaScript directly and the notebook remains a tutorial instead of a code generator.

The source is organized into these sections:

| Section | What it does |
| --- | --- |
| Configuration | Loads `.env`, table name, index flag, and optional LLM model |
| Demo embeddings | Implements the LangChain.js `Embeddings` interface without an external API key |
| Demo corpus | Creates short LangChain documents with stable JSON metadata |
| Database helpers | Runs schema preflight checks and table inspection queries |
| OracleVS setup | Creates and populates the Oracle vector store |
| Retrieval | Runs similarity search, metadata filtering, and MMR |
| Answer generation | Sends retrieved context to a chat model when an API key is available |
| Main runner | Produces `rag_results.json` for the notebook to display |

In [3]:
source = NODE_APP.read_text(encoding="utf-8")
print(f"langchain_rag.mjs has {len(source.splitlines())} lines.")

markers = [
    "Configuration",
    "Demo embeddings",
    "Demo corpus and answer generation",
    "Database helpers",
    "OracleVS + retrieval",
    "End-to-end runner",
]
for marker in markers:
    print(f"{marker}:", "found" if marker in source else "not found")

langchain_rag.mjs has 602 lines.
Configuration: found
Demo embeddings: found
Demo corpus and answer generation: found
Database helpers: found
OracleVS + retrieval: found
End-to-end runner: found


## 4. Install the JavaScript Dependencies

`package.json` installs the packages used by the runnable JavaScript implementation:

- `@oracle/langchain-oracledb` for `OracleVS` and Oracle integration helpers
- `@langchain/core` for documents and embedding interfaces
- `@langchain/openai` for the optional final answer generation step
- `oracledb` for Node.js database connectivity
- `dotenv` for local configuration

In [4]:
subprocess.run([NPM_BIN, "install"], cwd=PROJECT_DIR, check=True)

CompletedProcess(args=['npm.cmd', 'install'], returncode=0)

## 5. Validate the JavaScript Syntax

Before touching the database, ask Node.js to parse the source file. This catches syntax errors independently from Oracle credentials or network connectivity.

In [5]:
subprocess.run(["node", "--check", str(NODE_APP)], cwd=PROJECT_DIR, check=True)
print("JavaScript syntax check passed.")

JavaScript syntax check passed.


## 6. Run the End-to-End JavaScript Workflow

This cell runs the actual LangChain.js demo.

A successful run means the JavaScript application:

1. Opened a `node-oracledb` pool.
2. Validated basic schema table operations.
3. Reset the demo OracleVS table for deterministic reruns.
4. Created and populated the vector store from LangChain documents.
5. Ran similarity search, metadata-filtered search, and MMR retrieval.
6. Inspected the generated Oracle table.
7. Wrote a notebook-friendly result payload to `rag_results.json`.

If `OPENAI_API_KEY` is set, the final step also generates a grounded answer with a chat model. If not, the workflow still returns the retrieved context.

In [6]:
result = subprocess.run(
    [NPM_BIN, "run", "demo"],
    cwd=PROJECT_DIR,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("JavaScript demo failed. Check .env credentials and Oracle connectivity.")

if not RESULTS_JSON.exists():
    raise FileNotFoundError("The demo completed but did not write rag_results.json")


> langchain-js-rag-oracle-ai-database@1.0.0 demo
> node langchain_rag.mjs

Workflow completed: 7 documents ingested into LC_JS_RAG_DEMO; 4 similarity results, 2 filtered results, 3 MMR results.



## 7. Review the Preflight Summary

The preflight summary is important because it separates environment issues from LangChain issues. If the schema cannot create and drop a normal table, `OracleVS` cannot create its vector-store table either.

In [7]:
payload = json.loads(RESULTS_JSON.read_text(encoding="utf-8"))

rows = ["| Step | Status | Detail |", "| --- | --- | --- |"]
for item in payload.get("summary", []):
    detail = str(item["detail"]).replace("|", "\\|")
    rows.append(f"| `{item['step']}` | **{item['status']}** | {detail} |")
display(Markdown("\n".join(rows)))

| Step | Status | Detail |
| --- | --- | --- |
| `connect` | **PASS** | Connected with node-oracledb |
| `table_operations` | **PASS** | Created, inserted, selected 1 row, and dropped a test table |
| `reset_vector_store` | **PASS** | Dropped existing LC_JS_RAG_DEMO |
| `tablespace_quota` | **PASS** | USERS: unlimited quota |
| `oraclevs_ingest` | **PASS** | Inserted 7 documents into LC_JS_RAG_DEMO |
| `vector_index` | **READY** | Set LANGCHAIN_JS_CREATE_INDEX=true to create an HNSW vector index when supported |
| `similarity_search` | **PASS** | Retrieved 4 documents |
| `metadata_filter` | **PASS** | Retrieved 2 retrieval-category documents |
| `mmr_retrieval` | **PASS** | Selected 3 diverse documents |
| `rag_generation` | **PASS** | Generated a grounded answer with gpt-5.4 |

## 8. Inspect What OracleVS Created

A strong Oracle tutorial should prove that the vector store is inspectable. The JavaScript app queries Oracle data dictionary views after ingestion and returns the table shape, row count, and indexes.

This matters because OracleVS is not an opaque store. The database contains the source text, metadata, and vectors in normal schema objects that developers can inspect and govern.

In [8]:
inspection = payload["tableInspection"]

column_rows = ["| Column | Type |", "| --- | --- |"]
for column in inspection["columns"]:
    column_rows.append(f"| `{column['column']}` | `{column['type']}` |")

display(Markdown(f"**OracleVS row count:** `{inspection['rowCount']}`"))
display(Markdown("\n".join(column_rows)))

vector_indexes = [index for index in inspection["indexes"] if index.get("category") == "vector"]
internal_indexes = [index for index in inspection["indexes"] if index.get("category") != "vector"]

if vector_indexes:
    vector_index_rows = ["| Vector index | Type |", "| --- | --- |"]
    for index in vector_indexes:
        vector_index_rows.append(f"| `{index['index']}` | `{index['type']}` |")
    display(Markdown("\n".join(vector_index_rows)))
else:
    display(Markdown(
        "No vector index was reported. Set `LANGCHAIN_JS_CREATE_INDEX=true` when your database environment supports vector index creation."
    ))

if internal_indexes:
    internal_index_rows = ["| Internal/system index | Type |", "| --- | --- |"]
    for index in internal_indexes:
        internal_index_rows.append(f"| `{index['index']}` | `{index['type']}` |")
    display(Markdown("\n".join(internal_index_rows)))

**OracleVS row count:** `7`

| Column | Type |
| --- | --- |
| `ID` | `RAW` |
| `EXTERNAL_ID` | `VARCHAR2` |
| `EMBEDDING` | `VECTOR` |
| `TEXT` | `CLOB` |
| `METADATA` | `JSON` |

No vector index was reported. Set `LANGCHAIN_JS_CREATE_INDEX=true` when your database environment supports vector index creation.

| Internal/system index | Type |
| --- | --- |
| `SYS_C0010685` | `NORMAL` |
| `SYS_C0010686` | `NORMAL` |
| `SYS_IL0000133990C00003$$` | `LOB` |
| `SYS_IL0000133990C00004$$` | `LOB` |
| `SYS_IL0000133990C00005$$` | `LOB` |

## 9. Review the Demo Corpus and Metadata Contract

The corpus is small on purpose, but each document has metadata fields that represent the kind of contract production RAG systems usually need:

- `source` for provenance
- `category` for retrieval scope
- `product` for product-level filtering
- `priority` and `audience` for application-specific policy decisions

Metadata filters are retrieval boundaries. They decide which rows are eligible before context is assembled for the model.

In [9]:
rows = ["| # | Source | Category | Product | Snippet |", "| --- | --- | --- | --- | --- |"]
for index, doc in enumerate(payload["corpus"], start=1):
    metadata = doc["metadata"]
    snippet = doc["pageContent"].replace("|", "\\|")
    rows.append(
        f"| {index} | `{metadata['source']}` | `{metadata['category']}` | `{metadata['product']}` | {snippet} |"
    )
display(Markdown("\n".join(rows)))

| # | Source | Category | Product | Snippet |
| --- | --- | --- | --- | --- |
| 1 | `oracle-ai-database` | `database` | `Oracle AI Database` | Oracle AI Database stores vectors alongside operational data, so RAG applications can keep retrieval close to governed enterprise records. |
| 2 | `langchain-js` | `framework` | `LangChain.js` | LangChain.js RAG applications use OracleVS with Oracle AI Database as a vector store for adding documents, preserving metadata, and running similarity search retrieval. |
| 3 | `freesql` | `setup` | `FreeSQL` | FreeSQL gives developers a hosted Oracle Database schema for tutorials that only need normal schema table operations. |
| 4 | `metadata-filtering` | `retrieval` | `OracleVS` | Metadata filters narrow OracleVS retrieval by product, source, category, tenant, priority, or other JSON metadata fields. |
| 5 | `mmr` | `retrieval` | `LangChain.js` | Maximal marginal relevance helps LangChain.js retrieve a diverse context set instead of returning several near-duplicate chunks. |
| 6 | `embedding-provider` | `architecture` | `LangChain.js` | A production RAG service can swap the demo embedding class for OpenAI, OCI Generative AI, or Oracle-backed embeddings without changing OracleVS retrieval code. |
| 7 | `connection-options` | `setup` | `node-oracledb` | Autonomous AI Database and local Oracle Database use the same node-oracledb connection pattern when the schema can create and manage application tables. |

## 10. Compare Retrieval Modes

The same OracleVS table supports several retrieval modes through LangChain.js.

| Mode | What it proves |
| --- | --- |
| Similarity search | Baseline vector retrieval for a user question |
| Metadata-filtered search | JSON metadata can restrict the candidate set |
| MMR retrieval | LangChain can select diverse context instead of repeated near-duplicates |

This is the core RAG demonstration: documents go into Oracle AI Database once, then different LangChain retrieval strategies can use the same stored vectors and metadata.

In [10]:
def markdown_rows(title, rows):
    table = [f"### {title}", "", "| Rank | Distance | Source | Category | Snippet |", "| --- | --- | --- | --- | --- |"]
    for row in rows:
        distance = row.get("distance")
        distance_text = "n/a" if distance is None else f"{distance:.6f}"
        snippet = row["snippet"].replace("|", "\\|")
        table.append(
            f"| {row['rank']} | `{distance_text}` | `{row['source']}` | `{row['category']}` | {snippet} |"
        )
    return "\n".join(table)

display(Markdown(markdown_rows("Similarity search", payload["searchResults"])))
display(Markdown(markdown_rows("Metadata-filtered search", payload["filteredResults"])))
display(Markdown(markdown_rows("MMR retrieval", payload["mmrResults"])))

### Similarity search

| Rank | Distance | Source | Category | Snippet |
| --- | --- | --- | --- | --- |
| 1 | `0.123145` | `oracle-ai-database` | `database` | Oracle AI Database stores vectors alongside operational data, so RAG applications can keep retrieval close to governed enterprise records. |
| 2 | `0.148913` | `embedding-provider` | `architecture` | A production RAG service can swap the demo embedding class for OpenAI, OCI Generative AI, or Oracle-backed embeddings without changing OracleVS retrieval code. |
| 3 | `0.153764` | `langchain-js` | `framework` | LangChain.js RAG applications use OracleVS with Oracle AI Database as a vector store for adding documents, preserving metadata, and running similarity search retrieval. |
| 4 | `0.172201` | `connection-options` | `setup` | Autonomous AI Database and local Oracle Database use the same node-oracledb connection pattern when the schema can create and manage application tables. |

### Metadata-filtered search

| Rank | Distance | Source | Category | Snippet |
| --- | --- | --- | --- | --- |
| 1 | `0.092271` | `metadata-filtering` | `retrieval` | Metadata filters narrow OracleVS retrieval by product, source, category, tenant, priority, or other JSON metadata fields. |
| 2 | `0.729846` | `mmr` | `retrieval` | Maximal marginal relevance helps LangChain.js retrieve a diverse context set instead of returning several near-duplicate chunks. |

### MMR retrieval

| Rank | Distance | Source | Category | Snippet |
| --- | --- | --- | --- | --- |
| 1 | `n/a` | `oracle-ai-database` | `database` | Oracle AI Database stores vectors alongside operational data, so RAG applications can keep retrieval close to governed enterprise records. |
| 2 | `n/a` | `mmr` | `retrieval` | Maximal marginal relevance helps LangChain.js retrieve a diverse context set instead of returning several near-duplicate chunks. |
| 3 | `n/a` | `embedding-provider` | `architecture` | A production RAG service can swap the demo embedding class for OpenAI, OCI Generative AI, or Oracle-backed embeddings without changing OracleVS retrieval code. |

## 11. Generate a Grounded Answer

The final RAG step turns retrieved context into an answer. The JavaScript app uses the similarity-search rows as context and asks the chat model to answer only from that context.

If no `OPENAI_API_KEY` is configured, this step is intentionally skipped and the app prints the retrieved context instead. That keeps the OracleVS and retrieval demonstration runnable without an LLM key.

In [11]:
display(Markdown(payload["ragAnswer"].replace("\n", "  \n")))

A JavaScript RAG application can use **LangChain.js** with **OracleVS** as the vector store backed by **Oracle AI Database**. In this setup, OracleVS supports adding documents, preserving metadata, and performing similarity-search retrieval for the RAG pipeline. **Oracle AI Database** stores vectors alongside operational data, which keeps retrieval close to governed enterprise records. For connectivity, **node-oracledb** uses the same connection pattern for **Autonomous AI Database** and local **Oracle Database** when the schema can create and manage the application tables.

## 12. Why This Maps to TypeScript

The notebook runs JavaScript because Node.js can execute `.mjs` files directly. That avoids a TypeScript compile step in a tutorial notebook.

A TypeScript service uses the same imports and APIs, then adds its normal `tsconfig.json` and build process:

```typescript
import {
  DistanceStrategy,
  OracleVS,
  VectorElementFormat,
  createIndex,
} from "@oracle/langchain-oracledb";
```

The core pattern is unchanged:

1. Create a `node-oracledb` pool.
2. Provide a LangChain-compatible embeddings implementation.
3. Pass LangChain documents into `OracleVS.fromDocuments`.
4. Use `similaritySearchWithScore`, metadata filters, or MMR retrieval.
5. Pass retrieved context to the LLM selected by the application.

## 13. Integration Surface Beyond This First Tutorial

This notebook focuses on the most direct LangChain.js RAG path: `OracleVS` as the vector store, Node.js as the application runtime, and Oracle AI Database as the storage and retrieval layer.

| Component | Purpose | Covered in this tutorial |
| --- | --- | --- |
| `OracleVS` | Store and query document vectors in Oracle AI Database | Runnable end to end |
| `DistanceStrategy` | Choose vector distance behavior such as cosine distance | Runnable end to end |
| `VectorElementFormat` | Control dense vector storage format | Runnable end to end |
| Metadata filters | Restrict retrieval through JSON metadata predicates | Runnable end to end |
| MMR retrieval | Return a diverse context set instead of near-duplicate chunks | Runnable end to end |
| `createIndex` | Create HNSW or IVF vector indexes when supported | Opt-in runtime path |
| `OracleDocLoader` | Load supported documents through Oracle AI Database | Extension path |
| `OracleTextSplitter` | Split text through Oracle text processing | Extension path |
| `OracleEmbeddings` | Generate embeddings through Oracle AI Database | Extension path |
| `OracleSummary` | Summarize text through Oracle AI Database | Extension path |

The extension path is where a production application can add database-backed document loading, Oracle text splitting, in-database embedding generation, summarization, and vector-index tuning.

## 14. Operational Notes

A production service should make a few deliberate choices before moving beyond the tutorial:

- Replace deterministic demo embeddings with a production embedding provider.
- Keep one embedding dimension per OracleVS table.
- Design metadata fields before the corpus grows.
- Treat metadata filters as retrieval controls, not as the only authorization layer.
- Use environment-specific table names when several demos share one schema.
- Enable vector indexes where quota, privileges, and workload justify them.
- Keep cleanup and retention behavior explicit instead of dropping production tables on every run.

## Learn More

- [Oracle LangChain JavaScript integration guide](https://docs.oracle.com/en/database/oracle/oracle-database/26/aintg/langchain-oracledb-integration-guide/langchain-javascript.html)
- [Oracle AI Database integrations overview](https://docs.oracle.com/en/database/oracle/oracle-database/26/aintg/integrations.html)
- [LangChain.js Oracle AI Database vector store docs](https://docs.langchain.com/oss/javascript/integrations/vectorstores/oracleai)
- [npm: @oracle/langchain-oracledb](https://www.npmjs.com/package/@oracle/langchain-oracledb)
- [Source: oracle/langchain-oracle](https://github.com/oracle/langchain-oracle/tree/main/libs/js/langchain-oracledb)

## What This Proves

This notebook validates the first tutorial path for the LangChain TypeScript and JavaScript integration:

- `@oracle/langchain-oracledb` can create and populate an Oracle AI Database vector table from a JavaScript application.
- `OracleVS` supports document storage, semantic retrieval, metadata filtering, vector-index setup, and MMR retrieval.
- FreeSQL, Autonomous AI Database, or a local Oracle Database can be used when the schema supports the required table operations.
- The retrieved OracleVS context can be passed to a chat model to generate a grounded RAG answer.
- The same pattern can move to TypeScript, production embeddings, Autonomous AI Database, or local Oracle Database without changing the core LangChain RAG flow.

The next useful tutorials would go deeper on ingestion with `OracleDocLoader` and `OracleTextSplitter`, plus production embedding and summarization examples with `OracleEmbeddings` and `OracleSummary`.